# 02 · The latency/accuracy tradeoff

The headline result. The offline reference is **not a competitor** — it is the infinite-latency asymptote. What is being measured is the cost of a bounded emission delay, and where that cost stops mattering.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
os.environ.setdefault("OMP_NUM_THREADS", "2")

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch; torch.set_num_threads(2)

TABLES = os.path.join("..", "results", "tables")
def table(name):
    return pd.read_csv(os.path.join(TABLES, name))
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 60)


In [ ]:
curve = table('latency_curve.csv')
online = curve[(curve.method == 'online') & np.isfinite(curve.latency_budget_ms)]
online = online.sort_values('latency_budget_ms')
cols = ['latency_budget_ms', 'der', 'der_noise_scale', 'confusion',
        'n_speakers_pred', 'latency_median_ms', 'extra_mean_query_size']
print(online[[c for c in cols if c in online.columns]].to_string(index=False))

The curve is **flat below one hop** by construction: a budget of `B` ms buys `floor(B / 250)` hops of lookahead, so 0 ms and 125 ms are the same system. Both are in the sweep so that step is visible rather than assumed away.

In [ ]:
offline = curve[curve.method.str.startswith('offline')]
cols = ['method', 'der', 'confusion', 'latency_median_ms']
print(offline[cols].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
x = online.latency_budget_ms.to_numpy(float); y = online.der.to_numpy(float)
b = np.nan_to_num(online.der_noise_scale.to_numpy(float))
axes[0].plot(x, y, 'o-', lw=2, color='#1f77b4', label='online')
axes[0].fill_between(x, y - b, y + b, alpha=0.18, color='#1f77b4',
                     label=r'$\pm\sqrt{2}\sigma_{seed}$')
for _, r in offline.iterrows():
    axes[0].axhline(r.der, ls='--', lw=1.5,
                    color='#2ca02c' if 'ahc' in r.method else '#9467bd',
                    label=f"{r.method} (offline)")
axes[0].set_xscale('symlog', linthresh=125)
axes[0].set_xlabel('latency budget (ms)'); axes[0].set_ylabel('DER')
axes[0].set_title('DER vs latency budget'); axes[0].grid(alpha=0.3)
axes[0].legend(fontsize=7.5)
axes[1].plot(np.maximum(online.latency_median_ms, 1),
         y, 'o-', lw=2, color='#1f77b4')
for _, r in offline.iterrows():
    axes[1].plot(r.latency_median_ms, r.der, 'D', ms=10,
                 color='#2ca02c' if 'ahc' in r.method else '#9467bd',
                 label=r.method)
axes[1].set_xscale('log'); axes[1].set_xlabel('measured median emission delay (ms)')
axes[1].set_ylabel('DER')
axes[1].set_title('...vs what it actually costs in delay')
axes[1].grid(alpha=0.3, which='both'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## Where is the gap, and is it real?

Every difference is placed against `sqrt(2) * seed_sd` — the run-to-run scale of a difference between two runs. Anything smaller is noise regardless of its p-value.

In [ ]:
summary = table('method_summary.csv')
cols = ['variant', 'der', 'der_seed_sd', 'der_noise_scale', 'miss',
        'false_alarm', 'confusion', 'jer', 'latency_median_ms', 'n_speakers_pred']
print(summary[[c for c in cols if c in summary.columns]].to_string(index=False))

In [ ]:
tests = table('statistical_tests.csv')
sub = tests[tests.metric == 'der']
print(sub[['name_b', 'mean_a', 'mean_b', 'delta', 'ci_lower', 'ci_upper',
           'p_value', 'p_adjusted', 'noise_scale', 'noise_ratio',
           'verdict']].to_string(index=False))

**Unit of analysis.** These paired tests are over the 24 test recordings of one seed, so they condition on *one trained model per method*. They say whether these two sets of weights differ on this test set — not whether the method is better. That is why the `noise_ratio` column, not `p_adjusted`, decides the verdict.